# 99 — Results & Qualitative Inspection

Visualise comparison table, per-source breakdown, and manually spot-check 50 predictions per model.

**Runtime:** CPU.

In [ ]:
import os, sys, json
os.chdir('/content/polygence'); sys.path.insert(0, 'src')
from prompt_classifier.config import load_config
cfg = load_config('configs/base.yaml')

In [ ]:
from IPython.display import Markdown
with open('reports/comparison.md') as f:
    display(Markdown(f.read()))

In [ ]:
import pandas as pd

# Per-source breakdown for each model
for m in ['m1', 'm2', 'm3', 'm4']:
    p = f'reports/{m}.json'
    if not os.path.exists(p): continue
    with open(p) as f:
        r = json.load(f)
    if 'per_source' not in r: continue
    df = pd.DataFrame(r['per_source']).T
    print(f'\n=== {m} per-source ===')
    print(df.to_string())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# FPR / FNR comparison bar chart
models, fprs, fnrs, f1s = [], [], [], []
for m in ['m1', 'm2', 'm3', 'm4']:
    p = f'reports/{m}.json'
    if not os.path.exists(p): continue
    with open(p) as f:
        r = json.load(f)
    models.append(m); fprs.append(r['fpr']); fnrs.append(r['fnr']); f1s.append(r['f1'])

x = np.arange(len(models))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, vals, title, target in zip(axes, [f1s, fprs, fnrs], ['F1 (higher=better)', 'FPR (target <0.05)', 'FNR (target <0.10)'], [None, 0.05, 0.10]):
    ax.bar(x, vals)
    ax.set_xticks(x); ax.set_xticklabels(models)
    ax.set_title(title)
    if target: ax.axhline(target, color='red', linestyle='--', label=f'target={target}')
    ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Qualitative spot-check: 50 random test predictions for each model
test = pd.read_parquet(cfg['data']['test_path'])
sample = test.sample(50, random_state=7)

# Load m1 as example (swap import for other models)
from prompt_classifier.models.m1_tfidf_lr import predict as m1_predict
preds, probs = m1_predict(sample['prompt'].tolist())
sample = sample.copy()
sample['m1_pred'] = preds; sample['m1_prob'] = probs.round(3)
sample['correct'] = sample['m1_pred'] == sample['y']

print('Errors:')
errors = sample[~sample['correct']]
for _, row in errors.iterrows():
    print(f'  [y={row.y} pred={row.m1_pred} p={row.m1_prob}] {row.prompt[:120]}')